# 05 · Cross-dataset agreement (Finding 3)
Train a classifier on dataset A, predict dataset B, compare with B's own labels.
Each comparison is *exact*, *compatible* (ancestor/descendant: a granularity difference) or a *conflict*.

In [ ]:
import itertools
import pandas as pd
import celltypist
import scanpy as sc
from celltype_rel.ontology import relation

CHOSEN = []
def load(ds):
    a = sc.read_h5ad(f"../data/processed/{ds}.h5ad")
    a.var_names = a.var.feature_name.astype(str)
    sc.pp.normalize_total(a, target_sum=1e4); sc.pp.log1p(a)
    return a

In [ ]:
rows = []
for a_id, b_id in itertools.permutations(CHOSEN, 2):
    A, B = load(a_id), load(b_id)
    model = celltypist.train(A, labels="cell_type_ontology_term_id", feature_selection=True)
    pred = celltypist.annotate(B, model=model).predicted_labels.predicted_labels
    rel = [relation(p, t) for p, t in zip(pred, B.obs.cell_type_ontology_term_id)]
    rows.append({"train": a_id, "test": b_id, **pd.Series(rel).value_counts(normalize=True).to_dict()})
agreement = pd.DataFrame(rows)
agreement.to_csv("../results/agreement.csv", index=False)
agreement